# 04 | Explainable Competitor Benchmarking

**Project:** Dubai Short-Term Rental Intelligence  
**Data source:** AirROI Dubai extract (independent third-party data; not Airbnb internal records)  
**Observation grain:** One listing per calendar month  
**Original source coverage:** March 2021–February 2026 (the execution derives the actual reporting period from the loaded tables)

## Business objective

Develop and independently audit a **transparent competitive-rate benchmarking framework** for short-term rental property managers. For each eligible listing-month, compare its reported average daily rate (ADR) with the median ADR of other active properties of the **same room type and bedroom count**, located nearby in the **same reporting month**.

**Decision supported:** Identify listings that merit **manual pricing review** and show the coverage behind their comparison. **Decision not supported:** Automatically infer a fair price, booking uplift, profitability, or an optimal rate. All findings refer only to the supplied AirROI sample.

## 1 | Analytical scope and matching policy

| Component | Definition |
|:--|:--|
| Subject eligibility | Active listing, positive finite AED ADR, known room type, nonnegative integer bedroom count, plausible Dubai-region coordinates |
| Matching attributes | Same reporting month, same room type, exact bedroom count |
| Primary search radius | 2 km great-circle distance |
| Fallback radius | Expand to 5 km **only if** the 2 km group has fewer than 10 other listings |
| Minimum peer group | 10 peers, excluding the subject listing |
| Price comparison | `premium_pct = 100 × (subject ADR / peer median ADR − 1)` |
| Review categories | Absolute premium <20%: within threshold; 20–<35%: review; ≥35%: large deviation |
| Rate metric | AirROI `native_avg_daily_rate`, currency AED; monthly observed values |

The 10-peer rule and deviation thresholds are **analyst-selected screening heuristics**. The existing engine uses an Earth-centered KD-tree for candidate retrieval followed by exact Haversine-distance checks. Group size is evidence of *sample coverage*, not statistical confidence or price accuracy. The five nearest displayed peer examples are **not** the complete set used for the benchmark median.

**Important exclusions:** Amenities, guest capacity, host quality, exact building, and day-level rate variation are **not** controlled for in this model. The benchmark therefore identifies **rate differences**, not mispricing.

**Snapshot control (independent audit, 10 October 2026):** Fixed figures in this narrative refer to the four original files identified by SHA-256 in `reports/source_audit.json`. Executable tables derive values at runtime. Review the narrative if replacing the source snapshot. Raw data and listing-level outputs remain private; conclusions describe this sample and provider estimates.

In [1]:
from pathlib import Path
from contextlib import redirect_stdout
from IPython.display import display, Markdown
import io
import runpy
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Accept execution from the project root or notebooks/.
current = Path.cwd().resolve()
project_root = next((p for p in (current, *current.parents)
                     if (p / 'src' / '02_build_peer_benchmarks.py').is_file()
                     and (p / 'data' / 'processed').is_dir()), None)
if project_root is None:
    raise FileNotFoundError('Project not found. Open Jupyter in the project root or notebooks/.')

processed = project_root / 'data' / 'processed'
reports = project_root / 'reports'
figures = reports / 'figures'
figures.mkdir(parents=True, exist_ok=True)  # Independent reproducibility fix.
required_inputs = ['fact_listing_month.csv']
for filename in required_inputs:
    if not (processed / filename).is_file():
        raise FileNotFoundError(f'Missing {filename}; run 03_data_preparation.ipynb first.')

pd.set_option('display.max_columns', 18)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Project:', project_root.name)
print('Benchmark source available:', (project_root / 'src' / '02_build_peer_benchmarks.py').is_file())

## 2 | Execute the version-controlled benchmark pipeline

The implementation lives in `src/02_build_peer_benchmarks.py`; keeping the algorithm in a script supports reuse in notebooks, SQL, and Power BI exports. This notebook provides **independent validation and business interpretation** of that script. Source files remain unchanged. Listing-level outputs are generated locally and excluded from version control by `.gitignore`.

In [2]:
engine = runpy.run_path(str(project_root / 'src' / '02_build_peer_benchmarks.py'))['run']
log_buffer = io.StringIO()
with redirect_stdout(log_buffer):
    benchmark, peer_examples = engine()
(reports / 'benchmark_pipeline_log.txt').write_text(log_buffer.getvalue(), encoding='utf-8')

fact = pd.read_csv(processed / 'fact_listing_month.csv', dtype={'listing_id': 'string'}, low_memory=False)
benchmark['listing_id'] = benchmark['listing_id'].astype('string')
peer_examples['listing_id'] = peer_examples['listing_id'].astype('string')
peer_examples['peer_listing_id'] = peer_examples['peer_listing_id'].astype('string')
latest_month = benchmark['month_date'].max()
latest = benchmark.loc[benchmark.month_date.eq(latest_month)].copy()
latest_fact = fact.loc[fact.month_date.eq(latest_month)].copy()
valid_latest = latest.loc[latest.comparison_status.eq('Eligible')].copy()

summary = pd.DataFrame({
    'Indicator': ['Historical listing-month rows', 'Unique historical listings',
                  'Months represented', 'Latest sampled listings',
                  'Latest active listings', 'Latest benchmarked listings'],
    'Value': [len(fact), fact.listing_id.nunique(), fact.month_date.nunique(),
              len(latest_fact), int(latest_fact.active.sum()), len(valid_latest)]
})
display(summary)
print('Reporting month:', latest_month)
print('Detailed run log: reports/benchmark_pipeline_log.txt')

## 3 | Data integrity and algebraic checks

A complete benchmark output must preserve **one row per source listing-month**, avoid double-counting, suppress unusable comparison metrics, and calculate the rate premium consistently. These checks cover the **entire historical output**, not just February 2026.

In [3]:
assert len(benchmark) == len(fact), 'Source/benchmark row count mismatch'
assert not benchmark[['listing_id','month_date']].isna().any().any(), 'Null benchmark key'
assert not benchmark.duplicated(['listing_id','month_date']).any(), 'Duplicate benchmark key'

aligned = benchmark.merge(
    fact[['listing_id','month_date','native_avg_daily_rate']],
    on=['listing_id','month_date'], how='left', validate='one_to_one', indicator=True
)
assert aligned['_merge'].eq('both').all(), 'Unmatched benchmark rows'
valid_all = aligned.comparison_status.eq('Eligible')
assert aligned.loc[valid_all, 'peer_count'].ge(10).all(), 'Peer minimum violated'
assert aligned.loc[valid_all, 'peer_median_aed'].gt(0).all(), 'Bad peer median'
assert aligned.loc[valid_all, 'radius_km'].isin([2,5]).all(), 'Unexpected radius'
assert aligned.loc[~valid_all, ['peer_median_aed','premium_pct']].isna().all().all(), 'Unassessed premium/median must be suppressed'
assert aligned.loc[~valid_all, 'review_signal'].eq('Unassessed').all(), 'Unassessed rows were flagged'
expected = 100 * (aligned.loc[valid_all, 'native_avg_daily_rate'] /
                  aligned.loc[valid_all, 'peer_median_aed'] - 1)
assert np.allclose(aligned.loc[valid_all, 'premium_pct'], expected, atol=1e-8)

premium = aligned.loc[valid_all, 'premium_pct'].abs().round(9)
expected_signal = np.select([premium >= 35, premium >= 20],
                            ['Large deviation','Review'], default='Within threshold')
# At floating-point boundaries (e.g. +/-20%), CSV serialization can round tiny
# differences to an apparent exact threshold. The *stored engine review_signal*
# is authoritative. A disagreement is permissible only within numeric tolerance.
actual_signal = aligned.loc[valid_all, 'review_signal'].to_numpy()
threshold_disagreement = actual_signal != expected_signal
boundary_distance = np.minimum(np.abs(premium.to_numpy() - 20),
                               np.abs(premium.to_numpy() - 35))
assert not threshold_disagreement.any(), (
    'Review signals disagree away from allowed numerical threshold boundaries'
)

# Explicitly audit the *exported CSV*, which Power BI and SQL will read.
roundtrip = pd.read_csv(processed/'fact_peer_benchmark.csv', low_memory=False)
roundtrip_valid = roundtrip.loc[roundtrip.comparison_status.eq('Eligible')].copy()
roundtrip_premium = roundtrip_valid.premium_pct.abs().round(9).to_numpy(float)
roundtrip_expected = np.select([roundtrip_premium >= 35, roundtrip_premium >= 20],
                               ['Large deviation','Review'], default='Within threshold')
roundtrip_disagreement = roundtrip_valid.review_signal.to_numpy() != roundtrip_expected
roundtrip_boundary = np.minimum(np.abs(roundtrip_premium-20), np.abs(roundtrip_premium-35))
assert not roundtrip_disagreement.any(), (
    'Persisted review label disagrees with premium outside floating-point tolerance'
)
roundtrip_edge_cases = int(roundtrip_disagreement.sum())
print('Numeric threshold boundary cases after CSV round-trip:', roundtrip_edge_cases)
print('Power BI must use persisted review_signal, not recalculate flags from rounded premium.')

checks = pd.DataFrame({'Check': [
    'Historical source rows preserved', 'No missing/duplicate listing-month keys',
    'All source keys matched', 'Eligible groups contain at least 10 peers',
    'Invalid comparisons suppressed', 'Premium formula and thresholds correct'],
    'Result': ['PASS'] * 6})
display(checks)

## 4 | Independent verification of the matching algorithm

The tests above confirm output integrity, but they do **not** establish that selected comparables are appropriate. To avoid circular validation, the following audit recomputes peers independently using a **brute-force, vectorized Haversine calculation**, rather than the engine's KD-tree search.

The audit checks **every independently eligible subject in the latest observed reporting month** and verifies the exact peer count, 2 km → 5 km fallback, median, quartiles, and premium. It additionally tests the latest-month peer examples against subject/peer metadata. Small differences below floating-point tolerance are accepted.

In [4]:
def independent_distance_km(lat, lon, other_lat, other_lon):
    """Brute-force Haversine distance; deliberately independent of the engine."""
    earth_km = 6371.0088
    a, b = np.deg2rad(lat), np.deg2rad(lon)
    c, d = np.deg2rad(np.asarray(other_lat, float)), np.deg2rad(np.asarray(other_lon, float))
    q = np.sin((c-a)/2)**2 + np.cos(a)*np.cos(c)*np.sin((d-b)/2)**2
    return 2*earth_km*np.arcsin(np.sqrt(np.clip(q, 0, 1)))

# Eligibility reconstructed from source columns, not imported from the engine.
active = latest_fact.active.astype(str).str.lower().isin(['true','1'])
bedrooms = pd.to_numeric(latest_fact.bedrooms, errors='coerce')
rates = pd.to_numeric(latest_fact.native_avg_daily_rate, errors='coerce')
independent_eligible = (
    active & latest_fact.room_type.notna() & bedrooms.notna() &
    bedrooms.ge(0) & bedrooms.mod(1).eq(0) &
    latest_fact.latitude.between(24.5, 25.6) &
    latest_fact.longitude.between(54.5, 56.0) &
    np.isfinite(rates) & rates.gt(0)
).fillna(False)
subjects = latest_fact.loc[independent_eligible].copy()
subject_index = latest.set_index('listing_id', verify_integrity=True)
# Source-derived eligibility validation: no snapshot-specific expected count.
# These two non-eligible/eligible status groups together must partition the latest month.
assert subjects.listing_id.is_unique, 'Eligible subject IDs must be unique'
assert latest_fact.listing_id.is_unique, 'Latest fact listing IDs must be unique'
engine_eligible = latest.loc[latest.comparison_status.isin(['Eligible', 'Insufficient peers'])]
assert set(subjects.listing_id) == set(engine_eligible.listing_id), (
    'Independent source eligibility and engine status disagree'
)
assert len(subjects) + int(latest.comparison_status.eq('Ineligible').sum()) == len(latest_fact), (
    'Eligibility and ineligibility do not cover the entire latest-month sample'
)
print('Source-derived matching-eligible subjects:', f'{len(subjects):,}')

independent_results = []
for row in subjects.itertuples(index=False):
    peers_pool = subjects.loc[
        subjects.room_type.eq(row.room_type) & subjects.bedrooms.eq(row.bedrooms) &
        subjects.listing_id.ne(row.listing_id)
    ].copy()
    km = independent_distance_km(row.latitude, row.longitude,
                                 peers_pool.latitude.to_numpy(), peers_pool.longitude.to_numpy())
    within_two = peers_pool.loc[km <= 2.0 + 1e-7]
    radius = 2 if len(within_two) >= 10 else 5
    chosen = within_two if radius == 2 else peers_pool.loc[km <= 5.0 + 1e-7]
    stored = subject_index.loc[row.listing_id]
    assert stored.comparison_status == ('Eligible' if len(chosen) >= 10 else 'Insufficient peers')
    assert int(stored.peer_count) == len(chosen), f'Peer count mismatch: {row.listing_id}'
    assert np.isclose(stored.radius_km, radius), f'Fallback mismatch: {row.listing_id}'
    if len(chosen) >= 10:
        peer_rates = chosen.native_avg_daily_rate.to_numpy(float)
        independent_median = float(np.median(peer_rates))
        assert np.isclose(stored.peer_median_aed, independent_median)
        assert np.isclose(stored.peer_p25_aed, np.quantile(peer_rates, .25))
        assert np.isclose(stored.peer_p75_aed, np.quantile(peer_rates, .75))
        assert np.isclose(stored.premium_pct, 100*(row.native_avg_daily_rate/independent_median - 1))
    else:
        assert pd.isna(stored.premium_pct)
    independent_results.append({'listing_id': row.listing_id, 'matches_2km': len(within_two),
                                'matches_5km': int((km <= 5.0 + 1e-7).sum()),
                                'selected_radius_km': radius, 'peer_count': len(chosen)})
independent_audit = pd.DataFrame(independent_results)

# All latest subjects that are not eligible must be marked Ineligible.
ineligible_ids = set(latest_fact.listing_id) - set(subjects.listing_id)
assert subject_index.loc[list(ineligible_ids), 'comparison_status'].eq('Ineligible').all()
print(f'Independent matching checks: PASS for {len(subjects):,} eligible subjects; '
      f'{len(ineligible_ids):,} ineligible subjects confirmed')
display(independent_audit.groupby('selected_radius_km').agg(
    subjects=('listing_id','size'), median_peer_count=('peer_count','median')).reset_index())

## 4A | Matching-policy edge cases (synthetic fixtures)

In addition to auditing every latest-month eligible subject, the following controlled cases test the **independent validator's interpretation** of distance boundaries, minimum peer count, and fallback behavior. These are synthetic fixtures, not Airbnb observations. They complement—but do not replace—the audit of the production engine against actual source rows.

In [5]:
EARTH_KM_AUDIT = 6371.0088

def example_policy(distances):
    """Expected 2 km / 5 km selection from *precomputed* candidate distances."""
    d = np.asarray(distances, dtype=float)
    inner = np.sum(d <= 2.0 + 1e-7)
    radius = 2 if inner >= 10 else 5
    n = int(np.sum(d <= radius + 1e-7))
    return radius, n, 'Eligible' if n >= 10 else 'Insufficient peers'

# One degree latitude is approximately 111 km; exact great-circle metric used to build fixtures.
origin_lat, origin_lon = 25.0, 55.0
def make_distance(km):
    latitude = origin_lat + np.rad2deg(km/EARTH_KM_AUDIT)
    return float(independent_distance_km(origin_lat, origin_lon,
                                         [latitude], [origin_lon])[0])

assert example_policy([make_distance(1.5)]*9 + [make_distance(2.0)]) == (2,10,'Eligible')
assert example_policy([make_distance(1.5)]*9 + [make_distance(3.0)]) == (5,10,'Eligible')
assert example_policy([make_distance(1.5)]*9 + [make_distance(5.0)]) == (5,10,'Eligible')
assert example_policy([make_distance(1.5)]*9 + [make_distance(5.01)]) == (5,9,'Insufficient peers')
assert example_policy([]) == (5,0,'Insufficient peers')
print('Independent policy fixtures: 5/5 PASS (2 km boundary, fallback, 5 km boundary, no peers).')

## 5 | Verify the nearest-peer explanation file

`peer_examples_latest.csv` contains **up to five nearby examples per benchmarked subject**, for interpretability. The examples are not the full comparison cohort and **must never be used to recompute the peer median**. Validate both their metadata and their reported distances independently.

The five nearest examples for a listing are **illustrative rows only**; they cannot be used to recompute the median of its full (10+) peer cohort. The audit below explicitly measures how different those two medians can be.

In [6]:
assert not peer_examples.empty
assert peer_examples.listing_id.ne(peer_examples.peer_listing_id).all()
assert peer_examples.groupby('listing_id').size().le(5).all()
assert not peer_examples.duplicated(['listing_id','peer_listing_id']).any()
assert peer_examples.month_date.eq(latest_month).all()
assert peer_examples.listing_id.isin(valid_latest.listing_id).all()
assert peer_examples.peer_listing_id.isin(subjects.listing_id).all()

coords = latest_fact.set_index('listing_id')
a = coords.loc[peer_examples.listing_id.to_numpy()]
b = coords.loc[peer_examples.peer_listing_id.to_numpy()]
computed_km = independent_distance_km(a.latitude.to_numpy(), a.longitude.to_numpy(),
                                      b.latitude.to_numpy(), b.longitude.to_numpy())
assert np.allclose(computed_km, peer_examples.distance_km.to_numpy(), atol=.00051)
assert (a.room_type.to_numpy() == b.room_type.to_numpy()).all()
assert (a.bedrooms.to_numpy() == b.bedrooms.to_numpy()).all()
assert np.all(computed_km <= 5.0 + 1e-7)
assert np.all(coords.loc[peer_examples.peer_listing_id.to_numpy(), 'active'].to_numpy())

radius = valid_latest.set_index('listing_id')['radius_km']
assert (computed_km <= radius.loc[peer_examples.listing_id.to_numpy()].to_numpy() + 1e-7).all()
assert np.allclose(b.native_avg_daily_rate.to_numpy(float), peer_examples.peer_adr_aed.to_numpy(float))
print(f'Nearest-peer explanation file: PASS ({len(peer_examples):,} rows; '
      f'{peer_examples.listing_id.nunique():,} subjects)')

## 5A | Benchmark explainability: nearest five are **not** the peer median

The core engine calculates `peer_median_aed` from **all eligible nearby peers** (at least 10). `peer_examples_latest.csv` intentionally contains **up to five nearest properties** per benchmarked subject for an interface preview. A median of those examples is **not** the median used to flag a listing. This section verifies export completeness and quantifies how misleading a five-example-only calculation might be. No row-level sample values are published here.

For the Power BI Competitive Pricing Studio, label the examples as *five nearest examples*, show the **full peer count** beside the validated full-group median, and never compute a benchmark using only the five examples.

In [7]:
example_stats = (peer_examples.groupby('listing_id')
                 .agg(example_count=('peer_listing_id','size'),
                      nearest_five_median_aed=('peer_adr_aed','median')))
peer_explain = valid_latest.set_index('listing_id')[
    ['peer_count','peer_median_aed','peer_p25_aed','peer_p75_aed','radius_km']
].join(example_stats, how='left', validate='one_to_one')
assert peer_explain.example_count.notna().all(), 'Some benchmarked subjects have no explanation rows'
assert (peer_explain.example_count.to_numpy() == np.minimum(peer_explain.peer_count.to_numpy(), 5)).all(), (
    'Nearest-example count must be min(5, total peer count) for every benchmarked listing'
)
assert peer_explain.peer_median_aed.gt(0).all()
peer_explain['example_share_pct'] = 100 * peer_explain.example_count / peer_explain.peer_count
peer_explain['example_median_gap_pct'] = 100 * (
    peer_explain.nearest_five_median_aed / peer_explain.peer_median_aed - 1
)
peer_explain['peer_iqr_ratio_pct'] = 100 * (
    (peer_explain.peer_p75_aed - peer_explain.peer_p25_aed) / peer_explain.peer_median_aed
)
assert np.isfinite(peer_explain[['example_share_pct', 'example_median_gap_pct', 'peer_iqr_ratio_pct']]).all().all()

peer_summary = pd.DataFrame([
    ('Benchmarked listings with an example set', len(peer_explain)),
    ('Nearest example rows', len(peer_examples)),
    ('Median share of full peer cohort illustrated (%)', peer_explain.example_share_pct.median()),
    ('Median absolute gap: five-example median vs full median (%)', peer_explain.example_median_gap_pct.abs().median()),
    ('Examples differing by >20% from full median (listings)',
     int(peer_explain.example_median_gap_pct.abs().gt(20).sum())),
], columns=['Metric','Value'])
display(peer_summary)

# Aggregate visualization only; axes are limited to a readable range, without dropping audit rows.
full_prices = peer_explain.peer_median_aed
example_prices = peer_explain.nearest_five_median_aed
upper = float(np.nanquantile(np.concatenate([full_prices,example_prices]),.98))
shown = full_prices.le(upper) & example_prices.le(upper)
fig, ax = plt.subplots(figsize=(7.3,5.5))
ax.scatter(full_prices[shown],example_prices[shown], alpha=.26, s=17, color='#237B8C')
ax.plot([0,upper], [0,upper], linestyle='--', color='#D66F4C', linewidth=1.7,
        label='Five-example median equals full-group median')
ax.set(xlabel='Validated median of ALL matched peers (AED)',
       ylabel='Median of five nearest examples (AED)',
       title='Why the five displayed peers cannot define the benchmark')
ax.set_xlim(0,upper); ax.set_ylim(0,upper)
ax.legend(frameon=False, fontsize=8, loc='upper left')
ax.text(.03,.03, f'{(~shown).sum():,} points hidden by display limits; included in all results',
        transform=ax.transAxes, fontsize=9)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
fig.savefig(figures/'04_five_examples_vs_full_median.png', dpi=160, bbox_inches='tight')
plt.show()

## 6 | Latest-period coverage and review signals

A flag is not an instruction to change a rate. An **unassessed** listing either lacks required input fields or does not have a sufficiently large comparison set. The comparison status and peer radius should remain visible in the Power BI Pricing Studio.

In [8]:
statuses = (latest.groupby(['comparison_status','review_signal'], dropna=False)
            .size().reset_index(name='listings')
            .sort_values(['comparison_status','review_signal']))
display(statuses)

n_active = int(latest_fact.active.sum())
n_valid = len(valid_latest)
n_insufficient = int(latest.comparison_status.eq('Insufficient peers').sum())
n_ineligible = int(latest.comparison_status.eq('Ineligible').sum())
is_review = valid_latest.review_signal.isin(['Review','Large deviation'])
n_review = int(is_review.sum())
n_high = int((is_review & valid_latest.premium_pct.gt(0)).sum())
n_low = int((is_review & valid_latest.premium_pct.lt(0)).sum())
assert n_review == n_high + n_low, 'Review directions do not reconcile'
coverage = pd.DataFrame([
    ('Ineligible / no comparison', n_ineligible),
    ('Eligible / insufficient peers', n_insufficient),
    ('Within ±20%', int(valid_latest.review_signal.eq('Within threshold').sum())),
    ('Review: 20%–<35%', int(valid_latest.review_signal.eq('Review').sum())),
    ('Large deviation: ≥35%', int(valid_latest.review_signal.eq('Large deviation').sum()))
], columns=['Status','Listings'])
assert coverage.Listings.sum() == len(latest)

fig, ax = plt.subplots(figsize=(9, 4.7))
bars = ax.barh(coverage.Status, coverage.Listings, color=['#8D99A6','#B4BBC3','#42A8A1','#E5A248','#C65C51'])
ax.invert_yaxis()
ax.bar_label(bars, padding=4)
ax.set(xlabel='Observed listings', title=f'Peer benchmark coverage | {latest_month[:7]}')
ax.spines[['top','right']].set_visible(False)
ax.set_xlim(0, coverage.Listings.max()*1.15)
fig.tight_layout()
fig.savefig(figures/'04_benchmark_coverage.png', dpi=160, bbox_inches='tight')
plt.show()
print(f'Benchmarked: {n_valid:,} / {len(latest):,} observed, '
      f'or {n_valid/n_active:.1%} of active listings.')
print(f'Manual-review flags: {n_review:,} ({n_high:,} above +20%; {n_low:,} below −20%).')

## 7 | Outlier investigation and robust interpretation

Median peer prices reduce the influence of exceptionally expensive **peers**, but an extreme **subject-to-peer premium** can still arise from property heterogeneity or unusual input values. The distribution plot restricts only the **displayed horizontal range**; observations outside that range are **retained in all calculations and counts**. This makes the chart readable without silently removing listings.

In [9]:
premiums = valid_latest.premium_pct.dropna()
quantiles = premiums.quantile([.05,.25,.5,.75,.95,.99])
price_distribution = pd.DataFrame({
    'Statistic':['5th percentile','25th percentile','Median','75th percentile','95th percentile','99th percentile','Minimum','Maximum'],
    'Premium (%)':[quantiles.loc[.05],quantiles.loc[.25],quantiles.loc[.5],quantiles.loc[.75],
                   quantiles.loc[.95],quantiles.loc[.99],premiums.min(),premiums.max()]})
display(price_distribution.round(1))

visible = premiums.between(-100, 200)
fig, ax = plt.subplots(figsize=(9,4.7))
ax.hist(premiums.loc[visible], bins=45, color='#236F86', alpha=.9, edgecolor='white', linewidth=.3)
for x in [-35, -20, 20, 35]:
    ax.axvline(x, color='#E2A34D' if abs(x)==20 else '#BF6555', linestyle='--', linewidth=1.1)
ax.axvline(premiums.median(), color='#183A56', linewidth=2, label=f'Median {premiums.median():+.1f}%')
ax.legend(loc='upper right', frameon=False)
ax.set(title='Rate premium relative to comparable listings | latest month',
       xlabel='Premium versus peer median (%)', ylabel='Properties shown')
ax.text(.01, .98, f'Display only: [-100%, +200%] | {int((~visible).sum()):,} outside view (not excluded)',
        transform=ax.transAxes, va='top', fontsize=9)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
fig.savefig(figures/'04_premium_distribution.png', dpi=160, bbox_inches='tight')
plt.show()

extreme_100 = int(premiums.abs().gt(100).sum())
extreme_200 = int(premiums.abs().gt(200).sum())
print(f'Absolute premium >100%: {extreme_100:,} listings; >200%: {extreme_200:,}.')
print('Extreme rates should be examined, not automatically labeled incorrect.')

## 8 | Radius fallback and sensitivity

The 5 km fallback improves peer coverage but may mix locations with different micro-market characteristics. The next comparison quantifies its use and **reconstructs the 2 km median for fallback subjects** where at least two nearby peers exist. The 2 km median with fewer than 10 peers is shown **only as a sensitivity diagnostic**, not as a valid benchmark.

In [10]:
radius_summary = valid_latest.groupby('radius_km').agg(
    benchmarked=('listing_id','size'), median_peers=('peer_count','median'),
    median_absolute_premium=('premium_pct', lambda v: float(np.median(np.abs(v))))
).reset_index()
display(radius_summary)

# Counterfactual 2 km medians for listings that needed a wider search.
fallback_ids = set(valid_latest.loc[valid_latest.radius_km.eq(5), 'listing_id'])
fall_back_diagnostics = []
for row in subjects.loc[subjects.listing_id.isin(fallback_ids)].itertuples(index=False):
    pool = subjects.loc[subjects.room_type.eq(row.room_type) &
                        subjects.bedrooms.eq(row.bedrooms) & subjects.listing_id.ne(row.listing_id)]
    dist = independent_distance_km(row.latitude,row.longitude,pool.latitude.to_numpy(),pool.longitude.to_numpy())
    near = pool.loc[dist <= 2 + 1e-7]
    stored = subject_index.loc[row.listing_id]
    assert len(near) < 10
    near_median = float(near.native_avg_daily_rate.median()) if len(near) >= 2 else np.nan
    diff_pct = 100*(stored.peer_median_aed / near_median - 1) if np.isfinite(near_median) and near_median > 0 else np.nan
    fall_back_diagnostics.append({'listing_id':row.listing_id, 'n_2km':len(near),
                                  'median_2km_diagnostic_aed':near_median,
                                  'benchmark_median_5km_aed':float(stored.peer_median_aed),
                                  'five_vs_two_median_diff_pct':diff_pct})
fallback_check = pd.DataFrame(fall_back_diagnostics)
print('Fallback subjects:',len(fallback_check))
print('With >=2 peers within 2 km:',int(fallback_check.five_vs_two_median_diff_pct.notna().sum()))
if fallback_check.five_vs_two_median_diff_pct.notna().any():
    delta = fallback_check.five_vs_two_median_diff_pct.dropna().abs()
    print(f'Absolute difference in 5 km vs diagnostic 2 km median: '
          f'median {delta.median():.1f}%; 90th percentile {delta.quantile(.9):.1f}%.')

fig, ax = plt.subplots(figsize=(7,4.3))
bars = ax.bar(['2 km peer group','5 km fallback'],
              [int(valid_latest.radius_km.eq(2).sum()), int(valid_latest.radius_km.eq(5).sum())],
              color=['#287F8B','#D9A451'])
ax.bar_label(bars, padding=4)
ax.set(ylabel='Benchmarked listings',title='How far the matching engine searches')
ax.spines[['top','right']].set_visible(False)
ax.set_ylim(0, max(b.get_height() for b in bars)*1.16)
fig.tight_layout()
fig.savefig(figures/'04_peer_radius_coverage.png', dpi=160, bbox_inches='tight')
plt.show()

## 8A | Interpreting peer-group evidence, not predicting price correctness

Having ten peers is a **coverage criterion**, not a proof of comparability. To document evidence strength without inventing a statistical confidence level, the following diagnostics distinguish:

- **Search radius:** A 5 km fallback can cross distinct micro-markets.
- **Peer count:** More peer observations can improve benchmark coverage, but not missing quality controls.
- **Peer-price dispersion:** `(peer P75 − peer P25) / peer median` highlights heterogeneous peer rates; **40% is a descriptive review threshold only**, not a validated confidence cut-off.

These indicators do not change the original algorithm or its 20%/35% review labels. A manager should investigate the characteristics of a selected property, review the source rates, and obtain actual booking data before recommending a rate change.

In [11]:
peer_explain['diagnostic_review_note'] = np.select(
    [peer_explain.radius_km.eq(5), peer_explain.peer_iqr_ratio_pct.gt(40)],
    ['Expanded geography: inspect location comparability',
     'Wide peer-rate dispersion: inspect property comparability'],
    default='Coverage criteria met; quality comparability remains unverified'
)
# Overlapping diagnostics are reported independently, not treated as mutually exclusive statuses.
coverage_diagnostics = pd.DataFrame([
    ('2 km selected radius', int(peer_explain.radius_km.eq(2).sum())),
    ('5 km fallback radius', int(peer_explain.radius_km.eq(5).sum())),
    ('Fewer than 20 peers', int(peer_explain.peer_count.lt(20).sum())),
    ('Peer IQR / median >40% (analyst flag)', int(peer_explain.peer_iqr_ratio_pct.gt(40).sum())),
    ('Both 5 km and high dispersion', int((peer_explain.radius_km.eq(5) &
                                            peer_explain.peer_iqr_ratio_pct.gt(40)).sum())),
], columns=['Indicator', 'Benchmark listings'])
display(coverage_diagnostics)

radius_diagnostics = peer_explain.groupby('radius_km').agg(
    subjects=('peer_count','size'),
    median_peer_count=('peer_count','median'),
    median_peer_iqr_over_median_pct=('peer_iqr_ratio_pct','median'),
    high_dispersion_share_pct=('peer_iqr_ratio_pct',lambda s:float(s.gt(40).mean()*100))
).reset_index()
display(radius_diagnostics.round(1))

# Assert that valid benchmarks have a correctly ordered interquartile range.
assert (peer_explain.peer_p25_aed <= peer_explain.peer_median_aed).all()
assert (peer_explain.peer_median_aed <= peer_explain.peer_p75_aed).all()
print('Peer coverage and dispersion diagnostics are observational; not a validated confidence score.')

## 9 | Audit summary and Power BI requirements

The outputs below are computed from the **current execution**. They serve as report-ready context without claiming that observed price deviations represent pricing mistakes. The notebook exports a small aggregated audit JSON; listing-level tables remain excluded from the public GitHub repository.

In [12]:
import hashlib
source_sha256 = hashlib.sha256((processed/'fact_listing_month.csv').read_bytes()).hexdigest()

# Add numerical evidence for how little of the full median is represented by nearest examples.
audited_latest = {
    'processed_fact_sha256':source_sha256,
    'csv_roundtrip_threshold_edge_cases_historical':roundtrip_edge_cases,
    'median_example_share_of_all_peers_pct':float(peer_explain.example_share_pct.median()),
    'median_abs_five_example_vs_full_median_gap_pct':float(peer_explain.example_median_gap_pct.abs().median()),
    'five_example_median_deviations_over_20pct':int(peer_explain.example_median_gap_pct.abs().gt(20).sum()),
    'peer_price_iqr_over_median_gt_40pct':int(peer_explain.peer_iqr_ratio_pct.gt(40).sum()),
    'reporting_month':latest_month,
    'historical_listing_month_rows':int(len(fact)),
    'latest_observed_listings':int(len(latest)),
    'latest_active_listings':n_active,
    'latest_matching_eligible':int(len(subjects)),
    'latest_valid_benchmarks':n_valid,
    'latest_ineligible':n_ineligible,
    'latest_insufficient_peers':n_insufficient,
    'latest_review_flagged_abs_ge_20_pct':n_review,
    'latest_review_above_plus_20_pct':n_high,
    'latest_review_below_minus_20_pct':n_low,
    'latest_2km_benchmarks':int(valid_latest.radius_km.eq(2).sum()),
    'latest_5km_benchmarks':int(valid_latest.radius_km.eq(5).sum()),
    'latest_extreme_abs_gt_100_pct':extreme_100,
    'independently_audited_eligible_subjects':int(len(subjects)),
    'peer_example_rows_audited':int(len(peer_examples)),
    'rules':'Same month, room type and bedrooms; 2 km else 5 km; >=10 other active peers',
    'full_peer_cohort_vs_examples':'Median uses all matched peers; peer_examples_latest has only five nearest illustration rows',
    'interpretation':'Manual review screening; not optimal-price or realized-revenue estimate'
}
(reports/'benchmark_latest_verified_v14.json').write_text(json.dumps(audited_latest, indent=2), encoding='utf-8')

text = f"""### Verified findings | {latest_month[:7]}

The AirROI source contains **{len(fact):,} listing-month observations**. In the latest month, **{n_valid:,}** of **{len(latest):,}** sampled properties have a valid peer-price benchmark (**{n_valid/n_active:.1%} of active listings**). The remaining **{n_ineligible:,}** lack matching eligibility and **{n_insufficient:,}** have too few peers; neither group receives a pricing label.

**{n_review:,}** benchmarked listings differ from their peer median by at least **20%**: **{n_high:,}** have a positive premium and **{n_low:,}** a negative premium. The engine uses **2 km** for **{int(valid_latest.radius_km.eq(2).sum()):,}** properties and expands to **5 km** for **{int(valid_latest.radius_km.eq(5).sum()):,}**. **{extreme_100:,}** benchmarked properties have absolute rate premiums exceeding 100%, indicating that outlier review and comparability caveats must remain prominent.

**Independent verification:** A brute-force Haversine audit reproduced the peer-selection results for **all {len(subjects):,} eligible subjects in {latest_month[:7]}**, alongside checks of **{len(peer_examples):,}** explanation rows. Historical key integrity, premium arithmetic, and review thresholds also passed.

**Business interpretation:** These are *pricing-review candidates*, not confirmed mispriced properties. The sample is not a census of Dubai, and this model does not adjust for amenities, building class, or guest capacity. Provider-estimated occupancy and revenue do not demonstrate causal pricing effects.

**Explainability:** The five nearest exported example listings are not the full peer group. The median of those five differs from the validated group median by more than 20% for **{int(peer_explain.example_median_gap_pct.abs().gt(20).sum()):,}** benchmarked subjects. The median fraction of the group represented by the five examples is **{peer_explain.example_share_pct.median():.1f}%**. Expose this distinction visibly in the Power BI details page.

**Power BI handoff:** Display the peer median, subject ADR, signed premium, peer count, selected radius, comparison status, and review signal together. Suppress pricing assessments for unbenchmarked listings; retain a link to nearest-peer examples with an explicit note that they are a partial explanation set. The next notebook loads the analytical data model into SQLite."""
display(Markdown(text))
print('Aggregate audit:', (reports/'benchmark_latest_verified_v14.json').relative_to(project_root))

## 10 | Methodological limitations and next stage

**Data provenance:** The supplied AirROI export is a third-party short-term-rental sample, not platform transaction logs. Its latest month is **February 2026**, not a current live market view. Raw and row-level transformed data should remain local until redistribution rights are verified.

**Interpretation safeguards:** (1) ADR reflects provider data and may not equal realized nightly receipts; (2) matching on room type, bedrooms and distance cannot neutralize quality or amenity differences; (3) fallback radius changes comparison composition; (4) heuristics have not been validated against revenue outcomes; (5) occupancy, revenue and RevPAR are model-estimated, not audited transactions.

**Next stage — Notebook 05:** Populate SQLite and verify that listing-month facts, price benchmarks, dimensions, and review labels retain their keys and analytical meaning. In Power BI, the **Competitive Pricing Studio** should expose matching criteria and benchmark suppression rules beside every flagged listing.

**Reproducibility and future snapshots:** The notebook validates eligible subjects against source-derived eligibility, not a hardcoded February 2026 row count. All latest-month statistics are generated at execution time. Source hash and month are recorded in `reports/benchmark_latest_verified_v14.json`. The broader README and manually written overview still describe the original snapshot and should be updated if the input data changes.

**Explanation rules for dashboards:** `peer_count` and `peer_median_aed` refer to the **full matched peer cohort**. The separate five-nearest examples are partial illustrations and must never be labeled the cohort or used to reproduce `peer_median_aed`. High dispersion and fallback geographic radius are *review diagnostics*, not calibrated statistical confidence scores.

**Numerical boundary consistency:** Classifications use the absolute unrounded premium rounded to nine decimal percentage places before the 20%/35% comparisons. This handles binary floating-point noise at exact thresholds. SQL validation applies the same rule; display-rounded premiums must not be used to reclassify observations.